<a href="https://colab.research.google.com/github/ana-lys/cleanrlAIS2603/blob/master/docs/CleanRL_AIS_Demo_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CleanRL's Huggingface Integration Demo



[<img src="https://img.shields.io/badge/license-MIT-blue">](https://github.com/vwxyzjn/cleanrl)
[![tests](https://github.com/vwxyzjn/cleanrl/actions/workflows/tests.yaml/badge.svg)](https://github.com/vwxyzjn/cleanrl/actions/workflows/tests.yaml)
[![docs](https://img.shields.io/github/deployments/vwxyzjn/cleanrl/Production?label=docs&logo=vercel)](https://docs.cleanrl.dev/)
[<img src="https://img.shields.io/discord/767863440248143916?label=discord">](https://discord.gg/D6RCjA6sVT)
[<img src="https://img.shields.io/youtube/channel/views/UCDdC6BIFRI0jvcwuhi3aI6w?style=social">](https://www.youtube.com/channel/UCDdC6BIFRI0jvcwuhi3aI6w/videos)
[![Code style: black](https://img.shields.io/badge/code%20style-black-000000.svg)](https://github.com/psf/black)
[![Imports: isort](https://img.shields.io/badge/%20imports-isort-%231674b1?style=flat&labelColor=ef8336)](https://pycqa.github.io/isort/)
[<img src="https://img.shields.io/badge/%F0%9F%A4%97%20Models-Huggingface-F8D521">](https://huggingface.co/cleanrl)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vwxyzjn/cleanrl/blob/master/docs/get-started/CleanRL_Huggingface_Integration_Demo.ipynb)


CleanRL is a Deep Reinforcement Learning library that provides high-quality single-file implementation with research-friendly features. It now has has 🧪 experimental support for saving and loading models from 🤗 HuggingFace's [Model Hub](https://huggingface.co/models). This notebook is a preliminary demo.


* 💾 [GitHub Repo](https://github.com/vwxyzjn/cleanrl)
* 📜 [Documentation](https://docs.cleanrl.dev/)
* 🤗 [HuggingFace Model Hub](https://huggingface.co/cleanrl)
* 🔗 [Open RL Benchmark reports](https://wandb.ai/openrlbenchmark/openrlbenchmark/reportlist)



## Get Started

CleanRL can be installed via `pip`. Let's say we are interested in pulling the model for [`dqn_atari_jax.py`](https://github.com/vwxyzjn/cleanrl/blob/master/cleanrl/dqn_atari_jax.py), we can install the algorithm-variant-specific dependencies as follows:

In [ ]:
!git clone https://github.com/ana-lys/cleanrlAIS2603.git
%cd cleanrlAIS2603
!pip install -e .

In [3]:
import wandb
import os
from google.colab import userdata

# Retrieve API Key from Colab Secrets
try:
    os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
    wandb.login()
    print("W&B Login Successful")
except Exception as e:
    print("Could not find WANDB_API_KEY in Secrets. Please add it to track experiments.")


Could not find WANDB_API_KEY in Secrets. Please add it to track experiments.


3. The PPO Algorithm Classes
This block defines your Args, Agent, and helper functions.


In [ ]:
import random
import time
from dataclasses import dataclass
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import tyro
from torch.distributions.categorical import Categorical
from torch.utils.tensorboard import SummaryWriter

@dataclass
class Args:
    tag: str = "v2"
    exp_name: str = "ppo_colab"
    seed: int = 99
    torch_deterministic: bool = True
    cuda: bool = True
    track: bool = True
    wandb_project_name: str = "cleanRL"
    wandb_entity: str = None # Replace with your W&B username if needed

    # Algorithm specific
    env_id: str = "CartPole-v1"
    total_timesteps: int = 500000
    learning_rate: float = 5.0e-4
    num_envs: int = 4 # Reduced for standard Colab stability
    num_steps: int = 128
    anneal_lr: bool = True
    gamma: float = 0.99
    gae_lambda: float = 0.95
    num_minibatches: int = 4
    update_epochs: int = 4
    norm_adv: bool = True
    clip_coef: float = 0.2
    clip_vloss: bool = False
    ent_coef: float = 0.01
    vf_coef: float = 0.5
    max_grad_norm: float = 0.5
    target_kl: float = None

    # Runtime computed
    batch_size: int = 0
    minibatch_size: int = 0
    num_iterations: int = 0

def make_env(env_id, idx):
    def thunk():
        env = gym.make(env_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        return env
    return thunk

def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class Agent(nn.Module):
    def __init__(self, envs):
        super().__init__()
        obs_shape = np.array(envs.single_observation_space.shape).prod()
        self.critic = nn.Sequential(
            layer_init(nn.Linear(obs_shape, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 1), std=1.0),
        )
        self.actor = nn.Sequential(
            layer_init(nn.Linear(obs_shape, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, envs.single_action_space.n), std=1.0),
        )

    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None):
        logits = self.actor(x)
        probs = Categorical(logits=logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(x)

4. Training Loop Execution
This cell runs the actual training. You can modify hyperparameters here before running.

In [ ]:
# Initialize Args (you can pass list of strings to override defaults)
args = tyro.cli(Args, args=[])
args.batch_size = int(args.num_envs * args.num_steps)
args.minibatch_size = int(args.batch_size // args.num_minibatches)
args.num_iterations = args.total_timesteps // args.batch_size
run_name = f"PPO_{args.env_id}_{args.seed}_{int(time.time())}"

# Setup Logging
if args.track:
    wandb.init(
        project=args.wandb_project_name,
        entity=args.wandb_entity,
        sync_tensorboard=True,
        config=vars(args),
        name=run_name,
        monitor_gym=False,
        save_code=True,
    )

writer = SummaryWriter(f"runs/{run_name}")

# Seeding & Device
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")

# Env & Agent Setup
envs = gym.vector.SyncVectorEnv([make_env(args.env_id, i) for i in range(args.num_envs)])
agent = Agent(envs).to(device)
optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate, eps=1e-5)

# --- START TRAINING LOOP ---
# (Insert the for iteration loop here from your provided code)
# Note: Ensure 'next_obs' and storage tensors are initialized as in your snippet.

print(f"Starting training on {device}...")
# ... [Insert the rest of your loop logic here] ...

envs.close()
writer.close()
if args.track:
    wandb.finish()